# 06 — AMX Tech Monthly Revenue Forecasting
Phase 9 compares required baselines with exponential smoothing using chronological, expanding-window evaluation. Set `RERUN_FORECAST = True` to recompute from validated CSV data; the default loads persisted results.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from sentinel.forecasting import build_monthly_net_revenue

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RERUN_FORECAST = False

if RERUN_FORECAST:
    from sentinel.data.loader import read_csv_dataset
    from sentinel.eda.cleaning import clean_dataset
    from sentinel.forecasting import run_revenue_forecast

    tables, _ = clean_dataset(read_csv_dataset(PROJECT_ROOT / 'data/generated'))
    future, backtest, _, report = run_revenue_forecast(tables['sales'])
else:
    report = json.loads((PROJECT_ROOT / 'models/revenue_forecast_report.json').read_text(encoding='utf-8'))
    future = pd.read_csv(PROJECT_ROOT / 'models/revenue_forecast.csv', parse_dates=['month'])
    backtest = pd.read_csv(PROJECT_ROOT / 'models/revenue_forecast_backtest.csv', parse_dates=['month'])

sales = pd.read_csv(PROJECT_ROOT / 'data/generated/sales.csv')
monthly_revenue = build_monthly_net_revenue(sales)
report['company'], report['series'], report['protocol']

## Validation selects the method
All forecasts are one month ahead. Validation determines the method; the later test window measures the already-fixed selection.

In [ ]:
validation_metrics = pd.DataFrame(report['validation_metrics']).T
test_metrics = pd.DataFrame(report['test_metrics']).T
validation_metrics.round({'mae': 0, 'rmse': 0, 'wape': 4}), test_metrics.round({'mae': 0, 'rmse': 0, 'wape': 4})

In [ ]:
selected = report['selected_method']
selected_test = backtest[(backtest.phase == 'test') & (backtest.method == selected)]
figure = go.Figure()
figure.add_trace(go.Scatter(x=monthly_revenue.index, y=monthly_revenue.values, name='Observed net revenue', line={'color': '#2563eb'}))
figure.add_trace(go.Scatter(x=selected_test.month, y=selected_test.forecast_net_revenue, name='One-step test forecast', line={'color': '#f59e0b', 'dash': 'dash'}))
figure.add_trace(go.Scatter(x=future.month, y=future.forecast_net_revenue, name='Future point forecast', line={'color': '#16a34a', 'dash': 'dot'}, mode='lines+markers'))
figure.update_layout(title='AMX Tech monthly net-revenue forecast', xaxis_title='Month', yaxis_title='Net revenue', template='plotly_white')
figure

In [ ]:
future.assign(forecast_millions=future.forecast_net_revenue / 1_000_000).round({'forecast_net_revenue': 0, 'forecast_millions': 3})

## Interpretation boundary
January 2025 net revenue is forecast at about 26.23 million. The history contains only two annual cycles, so seasonal Holt-Winters, SARIMA, and neural models are not justified. These are point estimates without calibrated intervals; pricing, pipeline, market, and operational context should accompany any planning decision.